# Comprehensive Survival Analysis on Global Pancreatic Cancer Clinical and Risk Factors

## 1. Problem Formulation & Significance
Pancreatic cancer remains one of the most lethal malignancies worldwide, characterized by aggressive progression and a historically low five-year survival rate. Understanding the clinical, demographic, and genetic determinants of patient survival is paramount for improving prognosis and tailoring therapeutic strategies. 

This project aims to conduct a rigorous **Survival Analysis (Time-to-Event Analysis)** using a global clinical dataset. We will model the risk factors affecting patient survival times and evaluate the efficacy of various treatment modalities. 

### Roadmap of the Analysis:
* **Phase 1: Data Preprocessing & Cleaning** (Target: Handling missing values, data type casting, and baseline statistical summaries).
* **Phase 2: Non-Parametric Estimation** (Kaplan-Meier survival curves to estimate survival probabilities across cohorts).
* **Phase 3: Semi-Parametric Modeling & Hypothesis Testing** (Log-Rank tests to compare survival distributions and Cox Proportional Hazards regression to estimate hazard ratios).
* **Phase 4: Parametric Modeling** (Weibull Accelerated Failure Time (AFT) models to understand the baseline survival distribution).

## 2. Dataset Description & Provenance
The dataset used in this study is retrieved from Kaggle: 
[Pancreatic Cancer Global Clinical and Risk Factor Dataset](https://www.kaggle.com/datasets/zkskhurram/pancreatic-cancer-global-clinical-and-risk-factor/data).

It contains 2,000 patient records across 60 countries and 6 WHO regions, encompassing 51 distinct clinical, behavioral, and genetic features (spanning from 2015 to 2025). Key fields include survival time (`Survival_Months`), status (`Survived`), demographic indicators, and tumor characteristics.

In [1]:
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt

In [2]:
cancer_data = pd.read_csv("pancreatic_cancer_dataset.csv")

## 3. Data Loading & Initial Exploration
We begin by importing the necessary Python libraries for data manipulation and visualization, followed by an initial inspection of the dataset schema, dimensions, and missing values.

In [3]:
cancer_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 51 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Patient_ID            2000 non-null   object 
 1   Diagnosis_Year        2000 non-null   int64  
 2   Diagnosis_Date        2000 non-null   object 
 3   WHO_Region            2000 non-null   object 
 4   Country               2000 non-null   object 
 5   Age                   2000 non-null   int64  
 6   Gender                2000 non-null   object 
 7   Ethnicity             2000 non-null   object 
 8   Smoking_Status        2000 non-null   object 
 9   Pack_Years            2000 non-null   int64  
 10  Alcohol_Use           1414 non-null   object 
 11  BMI                   2000 non-null   float64
 12  BMI_Category          2000 non-null   object 
 13  Diabetes_Status       2000 non-null   object 
 14  Chronic_Pancreatitis  2000 non-null   object 
 15  Family_History       

## 4. Data Preprocessing & Cleaning Strategy
Initial exploration reveals missing data in several critical features. To maintain statistical integrity without discarding valuable data points, we apply the following imputation and transformation strategies:

* **Imputation of Categorical Variables**: Missing entries in `Alcohol_Use`, `Hereditary_Syndrome`, and `Vascular_Involvement` are filled with an explicitly designated `"Unknown"` class to preserve sample size ($N = 2,000$).
* **Surgical Margin Refinement**: Missing surgical margins are explicitly labeled as `"R3 (Unknown)"`.
* **Temporal Alignment**: The `Diagnosis_Date` column is converted from a generic string object to a proper `datetime64` format.
* **Target Variable Encoding**: The `Survived` column (originally encoded as "Yes"/"No") is mapped numerically (`{"No": 1, "Yes": 0}`) to signify the **event occurrence (death)**, which is the standard convention for survival analysis frameworks.

In [4]:
cancer_data["Alcohol_Use"] = cancer_data["Alcohol_Use"].fillna("Unknown")
cancer_data["Hereditary_Syndrome"] = cancer_data["Hereditary_Syndrome"].fillna("Unknown")
cancer_data["Surgical_Margin"] = cancer_data["Surgical_Margin"].fillna("R3 (Unknown)")
cancer_data["Vascular_Involvement"] = cancer_data["Vascular_Involvement"].fillna("Unknown")
cancer_data["Diagnosis_Date"] = pd.to_datetime(cancer_data["Diagnosis_Date"])

In [5]:
cancer_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 51 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Patient_ID            2000 non-null   object        
 1   Diagnosis_Year        2000 non-null   int64         
 2   Diagnosis_Date        2000 non-null   datetime64[ns]
 3   WHO_Region            2000 non-null   object        
 4   Country               2000 non-null   object        
 5   Age                   2000 non-null   int64         
 6   Gender                2000 non-null   object        
 7   Ethnicity             2000 non-null   object        
 8   Smoking_Status        2000 non-null   object        
 9   Pack_Years            2000 non-null   int64         
 10  Alcohol_Use           2000 non-null   object        
 11  BMI                   2000 non-null   float64       
 12  BMI_Category          2000 non-null   object        
 13  Diabetes_Status   

## 5. Preliminary Baseline & Exploratory Summary
Following data cleaning, we evaluate the baseline survival metrics and look into the dataset's composition. A key metric is the event distribution (deceased vs. censored/survived patients) to ensure the feasibility of the subsequent survival models.

In [6]:
cancer_data["Survived"] = cancer_data["Survived"].map({"No": 1, "Yes": 0})

In [7]:
event_counts = cancer_data["Survived"].value_counts()

deceased_count = event_counts.get(1, 0)
censored_count = event_counts.get(0, 0)

In [8]:
print(f"Dataset Composition: {deceased_count:_} Deceased (Events), {censored_count:_} Censored (Survived)")

Dataset Composition: 1_762 Deceased (Events), 238 Censored (Survived)


# <center>**Data Summary**</center>

In [9]:

print("-" * 110)
print("PANCREATIC CANCER GLOBAL DATASET — LOADED")
print("-" * 110)

print(f"Records         -> {cancer_data.shape[0]:_}")
print(f"Features        -> {cancer_data.shape[1]:_}")
print(f"Period          -> {cancer_data.Diagnosis_Year.min()} year – {cancer_data.Diagnosis_Year.max()} year")
print(f"WHO Regions     -> {cancer_data.WHO_Region.nunique()}")
print(f"Countries       -> {cancer_data.Country.nunique()}")
print(f"Cancer Types    -> {cancer_data.Cancer_Type.nunique()}")
print(f"Mutations       -> {cancer_data.Genetic_Mutation.nunique()}")
print(f"Treatments      -> {cancer_data.Treatment.nunique()}")
print(f"Stage IV        -> {(cancer_data.Cancer_Stage == "Stage IV").mean():.2%}")
print(f"Survival Rate   -> {(cancer_data.Survived == 0).mean():.2%}")

print("-" * 110)

--------------------------------------------------------------------------------------------------------------
PANCREATIC CANCER GLOBAL DATASET — LOADED
--------------------------------------------------------------------------------------------------------------
Records         -> 2_000
Features        -> 51
Period          -> 2015 year – 2025 year
WHO Regions     -> 6
Countries       -> 60
Cancer Types    -> 6
Mutations       -> 11
Treatments      -> 14
Stage IV        -> 50.70%
Survival Rate   -> 11.90%
--------------------------------------------------------------------------------------------------------------


In [10]:
cancer_data.to_csv("pancreatic_cancer_dataset_cleaned.csv", index=False)